# Predictive Circuit Coding Training

This notebook is the Colab-first training surface. It mounts Google Drive, clones the repo, links the processed dataset and artifacts from Drive, optionally materializes a runtime subset, then runs training and evaluation.

Default behavior is intentionally simple: use the full canonical dataset unless you explicitly turn subset selection back on.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')
repo_root = Path('/content/predictive-circuit-coding')
drive_root = Path('/content/drive/MyDrive/pcc_runs')
drive_data_root = drive_root / 'data' / 'allen_visual_behavior_neuropixels'
drive_artifacts_root = drive_root / 'artifacts'
drive_root.mkdir(parents=True, exist_ok=True)
drive_artifacts_root.mkdir(parents=True, exist_ok=True)
print('Drive mounted.')
print('Repo root:', repo_root)
print('Drive dataset root:', drive_data_root)
print('Drive artifacts root:', drive_artifacts_root)

In [ ]:
if not repo_root.exists():
    !git clone https://github.com/jacobposchl/predictive-circuit-coding.git {repo_root}
%cd {repo_root}
!pip install -e .[notebook]

In [ ]:
repo_dataset_root = repo_root / 'data' / 'allen_visual_behavior_neuropixels'
repo_artifacts_root = repo_root / 'artifacts'

assert drive_data_root.exists(), f'Missing Drive dataset bundle: {drive_data_root}'

repo_dataset_root.parent.mkdir(parents=True, exist_ok=True)
if repo_dataset_root.exists() or repo_dataset_root.is_symlink():
    if repo_dataset_root.is_symlink():
        repo_dataset_root.unlink()
    else:
        shutil.rmtree(repo_dataset_root)
repo_dataset_root.symlink_to(drive_data_root, target_is_directory=True)

if repo_artifacts_root.exists() or repo_artifacts_root.is_symlink():
    if repo_artifacts_root.is_symlink():
        repo_artifacts_root.unlink()
    else:
        shutil.rmtree(repo_artifacts_root)
repo_artifacts_root.symlink_to(drive_artifacts_root, target_is_directory=True)

print('Linked dataset into repo:', repo_dataset_root)
print('Linked artifacts into repo:', repo_artifacts_root)

In [ ]:
import subprocess
import yaml
from predictive_circuit_coding.utils import NotebookStageReporter, verify_paths_exist

reporter = NotebookStageReporter(name='colab-train', expected_duration='2-10 minutes for debug runs, longer for full A100 runs')
reporter.banner('Predictive Circuit Coding Training', subtitle='Setup, preflight, runtime selection, training, and evaluation')

base_experiment_config = repo_root / 'configs' / 'pcc' / 'predictive_circuit_coding_base.yaml'
data_config = repo_root / 'configs' / 'pcc' / 'allen_visual_behavior_neuropixels_local.yaml'
runtime_experiment_config = repo_root / 'artifacts' / 'colab_runtime_experiment.yaml'
checkpoint_dir = repo_root / 'artifacts' / 'checkpoints'
resume_checkpoint = checkpoint_dir / 'pcc_best.pt'

# Easiest default: use the full dataset. Set this to False only when you actively want
# the dataset_selection block from the base experiment config.
USE_FULL_DATASET = True

selection_disabled = {
    'output_name': 'full_dataset_colab',
    'session_ids': [],
    'subject_ids': [],
    'exclude_session_ids': [],
    'exclude_subject_ids': [],
    'session_ids_file': None,
    'subject_ids_file': None,
    'exclude_session_ids_file': None,
    'exclude_subject_ids_file': None,
    'experience_levels': [],
    'session_types': [],
    'image_sets': [],
    'session_numbers': [],
    'project_codes': [],
    'brain_regions_any': [],
    'min_n_units': None,
    'max_n_units': None,
    'min_trial_count': None,
    'max_trial_count': None,
    'min_duration_s': None,
    'max_duration_s': None,
    'split_seed': None,
    'split_primary_axis': None,
    'train_fraction': None,
    'valid_fraction': None,
    'discovery_fraction': None,
    'test_fraction': None,
}

experiment_payload = yaml.safe_load(base_experiment_config.read_text(encoding='utf-8'))
if USE_FULL_DATASET:
    experiment_payload['dataset_selection'] = selection_disabled

runtime_experiment_config.write_text(yaml.safe_dump(experiment_payload, sort_keys=False), encoding='utf-8')
experiment_config = runtime_experiment_config

dataset_selection_active = any(
    value not in (None, [], '', {})
    for key, value in experiment_payload.get('dataset_selection', {}).items()
    if key != 'output_name'
)

print('Experiment config:', experiment_config)
print('Data config:', data_config)
print('Runtime selection active:', dataset_selection_active)
print('Checkpoint path:', resume_checkpoint)

In [ ]:
reporter.begin('preflight', next_artifact='best checkpoint + evaluation summary')
paths_ok = verify_paths_exist({
    'experiment_config': experiment_config,
    'data_config': data_config,
    'drive_dataset_root': drive_data_root,
})
print(paths_ok)
assert all(paths_ok.values()), 'Missing config files or dataset bundle for training notebook.'

if dataset_selection_active:
    reporter.begin('runtime selection', next_artifact='selected split manifest')
    subprocess.run([
        'pcc-prepare-data',
        'materialize-runtime-selection',
        '--config', str(experiment_config),
        '--data-config', str(data_config),
    ], check=True)
    reporter.finish('runtime selection')
else:
    print('Using the full canonical dataset. No runtime subset will be materialized.')

print('If you want to resume, point training.resume_checkpoint in the base experiment config at an existing checkpoint.')
reporter.finish('preflight')

In [ ]:
reporter.begin('training', next_artifact='artifacts/checkpoints/pcc_best.pt')
%cd {repo_root}
subprocess.run([
    'pcc-train',
    '--config', str(experiment_config),
    '--data-config', str(data_config),
], check=True)
assert resume_checkpoint.exists(), f'Expected checkpoint was not written: {resume_checkpoint}'
reporter.note_checkpoint(resume_checkpoint)
reporter.finish('training')

In [ ]:
reporter.begin('evaluation', next_artifact='validation-ready evaluation summary json')
%cd {repo_root}
subprocess.run([
    'pcc-evaluate',
    '--config', str(experiment_config),
    '--data-config', str(data_config),
    '--checkpoint', str(resume_checkpoint),
    '--split', 'valid',
], check=True)
reporter.finish('evaluation')